# Notebook 07: CTF 入門

CTF（Capture The Flag）はセキュリティ技術を競うコンテストです。
実際の脆弱性を安全な環境で攻略し、隠された「フラグ」を見つけます。

## 学習目標
1. CTF の種類とジャンルを理解する
2. 基本的な CTF 問題を解いてみる
3. よく使うツールと考え方を身につける

---

**CTF は合法的にハッキング技術を学べる最良の場です**

## 1. CTF とは

### 形式
- **Jeopardy 形式**: ジャンル別の問題を解いてポイントを獲得（最も一般的）
- **Attack & Defense**: チームが自分のサーバーを守りつつ相手を攻撃
- **King of the Hill**: サーバーの管理権を奪い合う

### ジャンル
| ジャンル | 内容 | 例 |
|----------|------|----|
| Web | Web アプリの脆弱性 | SQLi, XSS, SSRF |
| Crypto | 暗号の解読 | RSA, AES, 古典暗号 |
| Forensics | データの解析 | ログ分析, メモリダンプ |
| Reversing | バイナリの解析 | 実行ファイルの逆アセンブル |
| Pwn | バイナリの脆弱性攻撃 | バッファオーバーフロー |
| OSINT | 公開情報の調査 | SNS, Google検索 |
| Misc | その他 | プログラミング, パズル |

### フラグの形式
```
FLAG{this_is_a_flag}
CTF{example_flag_here}
flag{some_text_123}
```

## 2. 練習問題: 暗号解読

まずは簡単な暗号解読から始めましょう。

In [ ]:
# === Challenge 1: Base64 デコード ===
# 難易度: ★☆☆☆☆

import base64

print("=== Challenge 1: Base64 ===")
encoded = "RkxBR3tiYXNlNjRfaXNfbm90X2VuY3J5cHRpb259"
print(f"暗号文: {encoded}")
print()
print("ヒント: Base64 エンコーディングは暗号化ではありません。")
print("       誰でもデコードできます。")
print()

# 解いてみよう！（下の行のコメントを外す）
# decoded = base64.b64decode(encoded).decode()
# print(f"フラグ: {decoded}")

In [ ]:
# === Challenge 2: シーザー暗号 ===
# 難易度: ★★☆☆☆

print("=== Challenge 2: シーザー暗号 ===")
ciphertext = "IODJ{fdhvdu_flskhu_lv_hdvb}"
print(f"暗号文: {ciphertext}")
print()
print("ヒント: シーザー暗号は各文字を一定数ずらす暗号です。")
print("       'IODJ' が 'FLAG' になるシフト数は？")
print()

def caesar_decrypt(text: str, shift: int) -> str:
    result = []
    for c in text:
        if c.isalpha():
            base = ord('A') if c.isupper() else ord('a')
            result.append(chr((ord(c) - base - shift) % 26 + base))
        else:
            result.append(c)
    return ''.join(result)

# 全シフト数を試す（ブルートフォース）
print("全パターン:")
for shift in range(1, 26):
    decrypted = caesar_decrypt(ciphertext, shift)
    if decrypted.startswith("FLAG"):
        print(f"  シフト {shift:2d}: {decrypted}  ← これ！")
    # else:
    #     print(f"  シフト {shift:2d}: {decrypted}")

In [ ]:
# === Challenge 3: Hex デコード ===
# 難易度: ★☆☆☆☆

print("=== Challenge 3: 16進数 ===")
hex_string = "464c41477b6865785f656e636f64696e677d"
print(f"暗号文: {hex_string}")
print()
print("ヒント: 2文字ずつ区切ると ASCII コードの16進数表現です。")
print("       46 = 'F', 4c = 'L', ...")
print()

# 解いてみよう！
# decoded = bytes.fromhex(hex_string).decode()
# print(f"フラグ: {decoded}")

In [ ]:
# === Challenge 4: XOR 暗号 ===
# 難易度: ★★★☆☆

print("=== Challenge 4: XOR 暗号 ===")

# 暗号化されたデータ（バイト列）
encrypted = bytes([0x27, 0x2f, 0x22, 0x24, 0x58, 0x5e, 0x2c, 0x31,
                   0x7e, 0x26, 0x30, 0x3a, 0x3e, 0x31, 0x2e, 0x58,
                   0x2a, 0x2c, 0x31, 0x58, 0x3e, 0x3c, 0x7e, 0x3e,
                   0x3a, 0x24, 0x5c])
print(f"暗号文（hex）: {encrypted.hex()}")
print()
print("ヒント: 鍵は 1 バイト。フラグは 'FLAG' で始まるはず。")
print("       'F' XOR key = 0x27 → key = ?")
print()

# 鍵の推測
key = encrypted[0] ^ ord('F')  # 'F' XOR 0x27 = ?
print(f"推測した鍵: 0x{key:02x} ('{chr(key)}')")
print()

# 復号
decrypted = bytes([b ^ key for b in encrypted]).decode()
print(f"フラグ: {decrypted}")

## 3. 練習問題: Web

Notebook 03 と 04 で学んだ SQL インジェクションと XSS を CTF 形式で復習します。

In [ ]:
# === Challenge 5: SQLi でフラグを取得 ===
# 難易度: ★★☆☆☆

import sqlite3

print("=== Challenge 5: SQLi CTF ===")
print()

# CTF 環境のセットアップ
conn = sqlite3.connect(":memory:")
c = conn.cursor()
c.execute("CREATE TABLE products (id INTEGER, name TEXT, price INTEGER)")
c.execute("INSERT INTO products VALUES (1, 'Apple', 100)")
c.execute("INSERT INTO products VALUES (2, 'Banana', 80)")
c.execute("CREATE TABLE secret_flags (id INTEGER, flag TEXT)")
c.execute("INSERT INTO secret_flags VALUES (1, 'FLAG{union_select_is_powerful}')")
conn.commit()

def search_products(query: str):
    """脆弱な商品検索（CTF問題）"""
    sql = f"SELECT id, name, price FROM products WHERE name LIKE '%{query}%'"
    try:
        return c.execute(sql).fetchall()
    except Exception as e:
        return f"Error: {e}"

print("この商品検索は SQLi に脆弱です。")
print("secret_flags テーブルからフラグを取得してください。")
print()
print("正常な検索:")
print(f"  search('Apple') = {search_products('Apple')}")
print()
print("ヒント: UNION SELECT を使ってみよう")
print("  カラム数は 3 つ (id, name, price)")
print()

# 解いてみよう！
# result = search_products("' UNION SELECT id, flag, 0 FROM secret_flags --")
# print(f"攻撃結果: {result}")

In [ ]:
# === Challenge 6: Forensics - 隠されたメッセージ ===
# 難易度: ★★☆☆☆

print("=== Challenge 6: 隠されたメッセージ ===")
print()

# 怪しいHTTPレスポンスヘッダー
headers = """HTTP/1.1 200 OK
Content-Type: text/html
X-Custom-1: 70
X-Custom-2: 76
X-Custom-3: 65
X-Custom-4: 71
X-Custom-5: 123
X-Custom-6: 104
X-Custom-7: 105
X-Custom-8: 100
X-Custom-9: 100
X-Custom-10: 101
X-Custom-11: 110
X-Custom-12: 95
X-Custom-13: 105
X-Custom-14: 110
X-Custom-15: 95
X-Custom-16: 104
X-Custom-17: 101
X-Custom-18: 97
X-Custom-19: 100
X-Custom-20: 101
X-Custom-21: 114
X-Custom-22: 115
X-Custom-23: 125
Server: nginx"""

print("以下のHTTPレスポンスヘッダーにフラグが隠されています:")
print(headers)
print()
print("ヒント: X-Custom-N ヘッダーの値は ASCII コード（10進数）です")
print()

# 解いてみよう！
values = []
for line in headers.split('\n'):
    if line.startswith('X-Custom-'):
        num = int(line.split(': ')[1])
        values.append(chr(num))
flag = ''.join(values)
print(f"フラグ: {flag}")

## 4. CTF に参加しよう

### 初心者向けプラットフォーム
- **CpawCTF**: 日本語の初心者向け CTF（常設）
- **picoCTF**: カーネギーメロン大学の初心者向け CTF
- **OverTheWire (Bandit)**: Linux コマンドの練習
- **KENRO**: 日本語の Web セキュリティ学習プラットフォーム

### CTF カレンダー
- **CTFtime.org**: 世界中の CTF イベントのスケジュール

### 便利ツール
| ツール | 用途 |
|--------|------|
| CyberChef | エンコード/デコード（ブラウザ上） |
| Burp Suite | HTTP プロキシ |
| Wireshark | パケット解析 |
| John the Ripper | パスワードクラック |
| nmap | ポートスキャン |
| Ghidra | リバースエンジニアリング |

## まとめ

CTF はゲーム感覚でセキュリティスキルを磨ける場です。

| ステップ | やること |
|----------|----------|
| 1 | このノートブックの問題を全て解く |
| 2 | CpawCTF や picoCTF に登録する |
| 3 | 簡単な問題から解き始める |
| 4 | Write-up（解法記事）を読んで学ぶ |
| 5 | チームを作って CTF イベントに参加 |

## 演習

1. Challenge 1-4 の暗号問題を全て解いてフラグを見つけよう
2. Challenge 5 の SQLi 問題を解いてみよう
3. labs/vulnerable_app を起動して、フラグ `FLAG{sql_injection_master}` を取得してみよう
4. CpawCTF に登録して Level 1 の問題に挑戦してみよう